Arthur Cox III. Dataset seed: 950531371; split/model seed: 42. Dataset: customer_360_C365-BD4DU.csv. All original code cells executed. Answers are in the accompanying PDF.


# COMP365 Customer 360 Classification Lab

## One notebook, three dataset sizes, one repeatable experiment

**Goal:** Use this notebook with **one** Customer 360 CSV at a time. Run it once with a 1K file, then again from the top with a 10K file, then with a 20K file. The code stays the same; only the uploaded file changes.

You will predict **`usage_trend_30d`** (`Declining`, `Stable`, or `Growing`) from customer, product, support, billing, and customer-success information. You will compare several classifiers, interpret their results, check a possible leakage issue in a separate renewal-stage exercise, and write down what the models can—and cannot—tell us.

> **Important boundary:** These are synthetic teaching data. A model finding a pattern here does not prove that the pattern is true for real customers, and it does not prove that one feature causes another.


## How to use this lab

1. In Google Colab, select **Runtime → Run all** (or work through the cells in order).
2. When prompted, upload **one** `customer_360_*.csv` file from one of the provided dataset bundles.
3. Record your results in the reflection section.
4. Start over from the top with a different size (or sample) file.

Keep `RANDOM_STATE = 42`. This makes the train/test split repeatable **for the file you chose**. Different files can still produce different results because they contain different synthetic customers.


## 0. Setup

In [ ]:
# Run this cell first. Colab already includes these common data-science packages.
import io
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.tree import DecisionTreeClassifier

sns.set_theme(style="whitegrid")
try:
    get_ipython().run_line_magic("matplotlib", "inline")
except NameError:
    pass
RANDOM_STATE = 42
TARGET = "usage_trend_30d"

try:
    from google.colab import files
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

print(f"Target: {TARGET}")
print(f"Running in Colab: {IN_COLAB}")


## 1. Upload one dataset

**What are we doing?** Loading a single CSV file for one experiment.

**Why?** A machine-learning result only has meaning when we know which data produced it. The notebook prints the exact filename, row count, and columns so you can record them.

**What to look for:** The expected target is `usage_trend_30d`, and the data should have 30 columns. If either check fails, stop and confirm you uploaded the correct Customer 360 CSV.


In [ ]:
# In Colab, this opens an upload picker. Do not change DATA_PATH in normal Colab use.
DATA_PATH = None if IN_COLAB else "customer_360_C365-BD4DU.csv"

if DATA_PATH is None:
    if not IN_COLAB:
        raise RuntimeError("This notebook is designed for Colab upload. Set DATA_PATH only for local testing.")
    uploaded = files.upload()
    if len(uploaded) != 1:
        raise ValueError("Please upload exactly one customer_360_*.csv file, then rerun this cell.")
    uploaded_name, uploaded_bytes = next(iter(uploaded.items()))
    if not uploaded_name.lower().endswith(".csv"):
        raise ValueError("Please upload a CSV file, not the ZIP bundle.")
    df = pd.read_csv(io.BytesIO(uploaded_bytes))
    source_name = uploaded_name
else:
    df = pd.read_csv(DATA_PATH)
    source_name = Path(DATA_PATH).name

if TARGET not in df.columns:
    raise ValueError(f"Expected target column '{TARGET}' was not found. Columns received: {list(df.columns)}")
if df.shape[1] != 30:
    print(f"Warning: expected 30 columns but found {df.shape[1]}. Review the schema below before continuing.")

print(f"Loaded: {source_name}")
print(f"Rows: {len(df):,} | Columns: {df.shape[1]}")
display(df.head())


## 2. Inspect the data before modeling

**What are we doing?** Checking column types, missing values, duplicate customer IDs, and the target-class balance.

**Why?** Models do not repair a bad question or a broken dataset. A missing satisfaction score, for example, is not automatically a mistake—it may mean the customer did not respond to a survey.

**Class balance matters:** If one class is much more common, a model can look accurate by mostly guessing that common class. That is why we will use macro precision, macro recall, and macro F1 later.


In [ ]:
print("Column types:")
display(pd.DataFrame({"data_type": df.dtypes.astype(str), "missing_values": df.isna().sum()}))

if "customer_id" in df.columns:
    print(f"Duplicate customer_id values: {df['customer_id'].duplicated().sum():,}")

target_counts = df[TARGET].value_counts(dropna=False)
target_share = (df[TARGET].value_counts(normalize=True, dropna=False) * 100).round(1)
display(pd.DataFrame({"count": target_counts, "percent": target_share}))

plt.figure(figsize=(7, 4))
sns.countplot(data=df, x=TARGET, order=target_counts.index, hue=TARGET, legend=False, palette="viridis")
plt.title("Target-class counts: usage_trend_30d")
plt.xlabel("Usage trend")
plt.ylabel("Customers")
plt.tight_layout()
plt.show()


### Checkpoint 1 — Before you build a model

Answer in your notes:

1. Which `usage_trend_30d` class appears most often?
2. Is any class rare enough that accuracy alone could be misleading? Why?
3. Which columns have missing values? Is a missing value necessarily an error?


## 3. Prepare the data fairly

**What are we doing?** We set aside a test set, remove identifiers, and create a reusable preprocessing pipeline.

**Why?**

- `customer_id` is an identifier, not a customer behavior. Letting a model use it could lead to memorization instead of learning a general pattern.
- We split before learning transformations so the test set remains a fair “unseen” check.
- Numeric columns receive median imputation and scaling. Categorical columns receive most-frequent imputation and one-hot encoding.
- `handle_unknown="ignore"` keeps the pipeline from crashing if a category appears in the test set but not the training set.

The data dictionary notes small, intentional capitalization differences in `industry`. The next cell normalizes that one known issue transparently instead of treating it as a new business signal.


In [ ]:
# Work on a copy so the original upload remains unchanged in this notebook session.
model_df = df.copy()

# Normalize the documented capitalization/spacing inconsistency in industry only.
if "industry" in model_df.columns:
    model_df["industry"] = (
        model_df["industry"].astype("string").str.strip().str.replace(r"\s+", " ", regex=True).str.title()
    )

# A missing target cannot be used to train or score a supervised classifier.
model_df = model_df.dropna(subset=[TARGET]).copy()

id_columns = [column for column in model_df.columns if column.lower() == "customer_id" or column.lower().endswith("_id")]
feature_columns = [column for column in model_df.columns if column not in id_columns + [TARGET]]

X = model_df[feature_columns]
y = model_df[TARGET]

class_counts = y.value_counts()
if class_counts.min() < 2:
    raise ValueError("Each target class needs at least 2 rows for a stratified train/test split.")

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=RANDOM_STATE, stratify=y
)

numeric_features = X_train.select_dtypes(include=np.number).columns.tolist()
categorical_features = [column for column in X_train.columns if column not in numeric_features]

numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])
categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("one_hot", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
])
preprocessor = ColumnTransformer([
    ("numeric", numeric_pipeline, numeric_features),
    ("categorical", categorical_pipeline, categorical_features),
], remainder="drop")

print(f"Excluded identifier columns: {id_columns}")
print(f"Features used: {len(feature_columns)} ({len(numeric_features)} numeric, {len(categorical_features)} categorical)")
print(f"Training rows: {len(X_train):,} | Test rows: {len(X_test):,}")


## 4. Compare classifiers

Each model receives the **same training rows, test rows, and preprocessing**. That makes the comparison fair.

- **Dummy baseline:** always predicts the most common class. This is not a serious model; it is the score a useful model should beat.
- **Logistic regression:** a simple linear classifier. It is often a strong, interpretable starting point.
- **Decision tree:** asks a series of if/then questions. It can model non-linear patterns but can overfit.
- **Random forest:** combines many randomized decision trees. It can capture more complex patterns and is usually less fragile than one tree.

Do not call the model with the highest accuracy “best” yet. First inspect the other metrics and the confusion matrices.


In [ ]:
models = {
    "Dummy baseline": DummyClassifier(strategy="most_frequent"),
    "Logistic regression": LogisticRegression(max_iter=2500, random_state=RANDOM_STATE),
    "Decision tree": DecisionTreeClassifier(min_samples_leaf=5, random_state=RANDOM_STATE),
    "Random forest": RandomForestClassifier(
        n_estimators=100, min_samples_leaf=2, random_state=RANDOM_STATE, n_jobs=-1
    ),
}

fitted_models = {}
predictions = {}
rows = []

for name, model in models.items():
    pipeline = Pipeline([("preprocess", clone(preprocessor)), ("model", model)])
    pipeline.fit(X_train, y_train)
    y_pred = pipeline.predict(X_test)
    fitted_models[name] = pipeline
    predictions[name] = y_pred
    rows.append({
        "model": name,
        "accuracy": accuracy_score(y_test, y_pred),
        "macro_precision": precision_score(y_test, y_pred, average="macro", zero_division=0),
        "macro_recall": recall_score(y_test, y_pred, average="macro", zero_division=0),
        "macro_f1": f1_score(y_test, y_pred, average="macro", zero_division=0),
    })

results = pd.DataFrame(rows).sort_values(["macro_f1", "accuracy"], ascending=False).reset_index(drop=True)
display(results.round(3))

metric_plot = results.melt(id_vars="model", value_vars=["accuracy", "macro_precision", "macro_recall", "macro_f1"],
                           var_name="metric", value_name="score")
plt.figure(figsize=(11, 5))
sns.barplot(data=metric_plot, x="model", y="score", hue="metric", palette="Set2")
plt.ylim(0, 1)
plt.xticks(rotation=15, ha="right")
plt.title("Model comparison on the same held-out test set")
plt.tight_layout()
plt.show()

champion_name = results.loc[0, "model"]
champion = fitted_models[champion_name]
champion_pred = predictions[champion_name]
print(f"Model selected for detailed inspection: {champion_name} (highest macro F1 in this run)")


## 5. Read the metrics in plain language

All scores below are calculated on the held-out test set: customers the model did not see during training.

| Metric | Plain-language question it answers | Why it matters here |
|---|---|---|
| Accuracy | “How often was the model right overall?” | Useful summary, but common classes can dominate it. |
| Precision | “When the model predicted a class, how often was that prediction right?” | Low precision means many false alarms for that class. |
| Recall | “Of the customers truly in a class, how many did the model find?” | Low recall means the model misses many real cases. |
| F1 | “How well did precision and recall work together?” | A balance of precision and recall. |
| Macro F1 | “What is the average F1 across classes, giving each class equal weight?” | Prevents a common class from hiding poor performance on smaller classes. |

There is no universally “best” metric. The best choice depends on the cost of being wrong. For example, a team that wants to find every declining-usage customer may care more about **recall** for `Declining` than overall accuracy.


In [ ]:
print(f"Detailed per-class report for: {champion_name}")
print(classification_report(y_test, champion_pred, zero_division=0))


## 6. Use confusion matrices to see the mistakes

Rows are the **actual** class. Columns are the model’s **predicted** class. Values on the diagonal are correct predictions. Values away from the diagonal show the kinds of mistakes a model made.

Look for a model that is highly accurate but repeatedly confuses a smaller class with a more common class. That is a reason to question a simple “highest accuracy wins” decision.


In [ ]:
labels = sorted(y.unique())
fig, axes = plt.subplots(2, 2, figsize=(12, 10))

for ax, (name, y_pred) in zip(axes.flat, predictions.items()):
    matrix = confusion_matrix(y_test, y_pred, labels=labels)
    sns.heatmap(matrix, annot=True, fmt="d", cmap="Blues", cbar=False,
                xticklabels=labels, yticklabels=labels, ax=ax)
    ax.set_title(name)
    ax.set_xlabel("Predicted")
    ax.set_ylabel("Actual")

plt.tight_layout()
plt.show()


### Checkpoint 2 — Choose a model with a reason

Answer in your notes:

1. Which model has the highest **accuracy**? Which has the highest **macro F1**? Are they the same?
2. Which class is hardest for the selected model to find? Use its **recall** and confusion-matrix row as evidence.
3. If the business most wanted to identify `Declining` usage, would you pick the same model? Explain using a metric, not a guess.
4. Did every serious model beat the dummy baseline? If not, what does that tell you?


## 7. Explain a model with permutation importance

**What are we doing?** We shuffle one feature in the test set at a time and measure how much the selected model’s macro F1 falls.

**Interpretation:** A large drop means the model relied on that feature **in this dataset and this model**. Near-zero or negative values mean shuffling it did not hurt much in this test.

**What this does not prove:** Importance is not causation. A feature can be important because it is a proxy, a consequence, or a record of the same underlying situation. For example, more CSM touchpoints could be important because struggling customers receive more outreach—not because outreach caused them to struggle.


In [ ]:
# Five repeats make the estimate less dependent on one shuffle while remaining practical for Colab.
importance = permutation_importance(
    champion, X_test, y_test, scoring="f1_macro", n_repeats=5,
    random_state=RANDOM_STATE, n_jobs=-1
)

importance_df = pd.DataFrame({
    "feature": X_test.columns,
    "mean_macro_f1_drop": importance.importances_mean,
    "std": importance.importances_std,
}).sort_values("mean_macro_f1_drop", ascending=False)

display(importance_df.head(12).round(4))

top_importance = importance_df.head(12).sort_values("mean_macro_f1_drop")
plt.figure(figsize=(9, 6))
plt.barh(top_importance["feature"], top_importance["mean_macro_f1_drop"], xerr=top_importance["std"], color="#4C78A8")
plt.xlabel("Drop in macro F1 after shuffling the feature")
plt.title(f"Permutation importance: {champion_name}")
plt.tight_layout()
plt.show()


### Checkpoint 3 — Importance is not causation

1. Name the top two features by permutation importance.
2. Write one **allowed** statement: “In this model and dataset, shuffling ___ lowered macro F1 by ___.”
3. Write one **not allowed** statement that would wrongly claim causation.
4. Give one alternative explanation for why the top feature might be important.


## 8. Renewal-stage leakage / ablation exercise

Now we change the question: predict **`renewal_stage`**, a current renewal-process context—not a renewal outcome. The column `days_until_renewal` may be extremely predictive because the stage could be defined or managed partly by how close the renewal date is.

This is a **leakage check**, not proof of leakage. The CSV and data dictionary alone do not prove exactly how the label was created. A very strong score with `days_until_renewal` is a reason to ask a data-governance question: *Was this feature known at the time of prediction, and is it too close to the label definition to teach the intended lesson?*

We will run the same model twice:

1. with `days_until_renewal`; and
2. after removing it (an **ablation**).

If performance falls sharply, that feature supplied a large share of the predictive signal. It does **not** prove the feature caused the renewal stage.


In [ ]:
LEAKAGE_TARGET = "renewal_stage"
LEAKAGE_CANDIDATE = "days_until_renewal"

if LEAKAGE_TARGET not in df.columns or LEAKAGE_CANDIDATE not in df.columns:
    print("Leakage/ablation exercise skipped: expected renewal columns were not found.")
else:
    renewal_df = df.dropna(subset=[LEAKAGE_TARGET]).copy()
    if "industry" in renewal_df.columns:
        renewal_df["industry"] = (
            renewal_df["industry"].astype("string").str.strip().str.replace(r"\s+", " ", regex=True).str.title()
        )

    # Do not use the first exercise's target as a predictor in this focused second exercise.
    renewal_ids = [column for column in renewal_df.columns if column.lower() == "customer_id" or column.lower().endswith("_id")]
    renewal_features = [
        column for column in renewal_df.columns
        if column not in renewal_ids + [LEAKAGE_TARGET, TARGET]
    ]
    X_renewal = renewal_df[renewal_features]
    y_renewal = renewal_df[LEAKAGE_TARGET]
    Xr_train, Xr_test, yr_train, yr_test = train_test_split(
        X_renewal, y_renewal, test_size=0.25, random_state=RANDOM_STATE, stratify=y_renewal
    )

    def make_renewal_pipeline(columns):
        numeric = Xr_train[columns].select_dtypes(include=np.number).columns.tolist()
        categorical = [column for column in columns if column not in numeric]
        prep = ColumnTransformer([
            ("numeric", Pipeline([("imputer", SimpleImputer(strategy="median"))]), numeric),
            ("categorical", Pipeline([
                ("imputer", SimpleImputer(strategy="most_frequent")),
                ("one_hot", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
            ]), categorical),
        ])
        return Pipeline([
            ("preprocess", prep),
            ("model", RandomForestClassifier(
                n_estimators=100, min_samples_leaf=2, random_state=RANDOM_STATE, n_jobs=-1
            )),
        ])

    ablation_rows = []
    for label, columns in {
        "All eligible features": renewal_features,
        f"Without {LEAKAGE_CANDIDATE}": [c for c in renewal_features if c != LEAKAGE_CANDIDATE],
    }.items():
        renewal_model = make_renewal_pipeline(columns)
        renewal_model.fit(Xr_train[columns], yr_train)
        renewal_pred = renewal_model.predict(Xr_test[columns])
        ablation_rows.append({
            "experiment": label,
            "accuracy": accuracy_score(yr_test, renewal_pred),
            "macro_f1": f1_score(yr_test, renewal_pred, average="macro", zero_division=0),
        })

    ablation_results = pd.DataFrame(ablation_rows)
    display(ablation_results.round(3))
    f1_drop = ablation_results.loc[0, "macro_f1"] - ablation_results.loc[1, "macro_f1"]
    print(f"Macro-F1 change after removing {LEAKAGE_CANDIDATE}: {f1_drop:+.3f}")


### Checkpoint 4 — Leakage and data-governance reasoning

1. How did accuracy and macro F1 change after removing `days_until_renewal`?
2. Does the change prove that `days_until_renewal` *causes* `renewal_stage`? Why not?
3. What question would you ask the data owner before using this model in a real decision?
4. If the business wanted an early-warning model six months before renewal, would `days_until_renewal` be acceptable, unavailable, or a possible shortcut? Defend your answer based on when the feature is known and how the label is defined.


## 9. Final reflection: compare your repeated runs

Complete this table after you run the notebook with one 1K, one 10K, and one 20K dataset. Use the same `RANDOM_STATE = 42` each time.

| Question | 1K dataset | 10K dataset | 20K dataset |
|---|---|---|---|
| Filename / sample |  |  |  |
| Best accuracy model and score |  |  |  |
| Best macro-F1 model and score |  |  |  |
| Class hardest to find |  |  |  |
| Top permutation-importance feature |  |  |  |
| Renewal-stage macro-F1 drop after ablation |  |  |  |

Then answer:

1. Did more rows make the results more stable or change the winning model? Use evidence from the table.
2. Did accuracy and macro F1 tell the same story in every run? If not, why does that matter?
3. What is one business decision this notebook should **not** make automatically?
4. State one conclusion supported by your results and one conclusion that would go beyond the evidence.

### Submission reminder

Record the dataset filename, its row count, the fixed `RANDOM_STATE`, the selected model, and the metrics you cite. These details make your work reproducible. Do not submit customer names, credentials, or any data outside the provided synthetic files.
